# 08 — Einsum for ML

**Workload:** Attention-style contractions, stable softmax, and batched matrix multiplication.

This notebook is executed against the RNP engine. Every output below is
stored in the notebook and visible when rendered on GitHub.

In [1]:
from pathlib import Path
import sys

# rnp is the Rust engine's numpy-compatible package: one import swap and
# everything below is ordinary NumPy code.
PROJECT_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "shim" / "rnp").is_dir()
)
sys.path.insert(0, str(PROJECT_ROOT / "shim"))

import rnp as np

probe = np.array(0)
print("rnp version:", np.__version__)
print(f"engine: {type(probe).__module__}.{type(probe).__name__}")
assert type(probe).__module__ == "_rnp"

rnp version: 2.5.2
engine: _rnp.ndarray


## Create a tiny attention batch

Seed query, key, and value tensors with deterministic Gaussian samples.

In [2]:
rng = np.random.default_rng(314159)
queries = rng.normal(size=(2, 3, 4))
keys = rng.normal(size=(2, 5, 4))
values = rng.normal(size=(2, 5, 3))
print("query/key/value shapes:", queries.shape, keys.shape, values.shape)

query/key/value shapes: (2, 3, 4) (2, 5, 4) (2, 5, 3)


## Compute stable attention

Contract queries with keys, normalize with softmax, and contract weights with values.

In [3]:
scores = np.einsum("bqd,bkd->bqk", queries, keys) / np.sqrt(queries.shape[-1])
shifted = scores - scores.max(axis=-1, keepdims=True)
weights = np.exp(shifted)
weights /= weights.sum(axis=-1, keepdims=True)
context = np.einsum("bqk,bkv->bqv", weights, values)
weight_sums = weights.sum(axis=-1)
print("attention row sums:\n", weight_sums)
print("first attention weights:", np.round(weights[0, 0], 6))
print("first context vector:", np.round(context[0, 0], 6))

attention row sums:
 [[1. 1. 1.]
 [1. 1. 1.]]
first attention weights: [0.311458 0.360885 0.127257 0.110798 0.089603]
first context vector: [-0.174672  0.521952 -0.060189]


## Cross-check with batched matrix multiplication

The score contraction has an equivalent batched `@` formulation.

In [4]:
batched_scores = queries @ keys.swapaxes(-1, -2) / 2.0
print("maximum score difference:", np.abs(scores - batched_scores).max())

maximum score difference: 2.220446049250313e-16


## Verify the result

In [5]:
assert np.allclose(weight_sums, np.ones((2, 3)), rtol=0.0, atol=1e-15)
assert np.allclose(scores, batched_scores, rtol=0.0, atol=1e-12)
expected_weights = [0.31145789, 0.36088471, 0.12725665, 0.11079817, 0.08960259]
assert np.allclose(weights[0, 0], expected_weights, rtol=0.0, atol=1e-8)
print("PASS — all attention assertions passed.")

PASS — all attention assertions passed.
